<table>
    <tr>
        <td><img src="https://s3.amazonaws.com/media-p.slid.es/uploads/1485763/images/9060062/Header.png" width="300"/></td>
        <td>&nbsp;</td>
        <td>
            <h1 style="font-size:200%;color:#6A0DAD;text-align:center"> <FONT COLOR="#6A0DAD"> Arquitecturas Avanzadas de Deep Learning</FONT> </h1></td>         
        <td>
            <tp><p style="font-size:99%;text-align:center">Sesión 7</p></tp>
            <tp><p style="font-size:115%;text-align:center">Junio 2026</p></tp>
            <tp><p style="font-size:115%;text-align:center">Prof. Daniel Rambaut </p></tp>
        </td>
    </tr>
</table>

En este notebook daremos un salto cualitativo importante: pasaremos de las **redes neuronales densas (MLP)** que vimos en la sesión anterior a las **arquitecturas especializadas** que dominan el panorama actual del *Deep Learning*. Cada una de estas arquitecturas fue diseñada para explotar la **estructura particular** de un tipo de dato (imágenes, secuencias, lenguaje).
<br>
Aprenderemos sobre:
- Las **Redes Convolucionales (CNNs)**, especializadas en **imágenes** y datos con estructura espacial.
- Las **Redes Recurrentes (RNNs / LSTM)**, especializadas en **secuencias** y **texto**.
- Los **Transformers**, la arquitectura detrás de **ChatGPT**, **BERT** y la revolución de la IA generativa.
- Técnicas de **regularización avanzada**: **Batch Normalization**, **Early Stopping** y **Data Augmentation**.

---
# <FONT SIZE=5 COLOR="green"> 0. Repaso de la sesión anterior </FONT>

En la sesión pasada construimos los fundamentos de las **Redes Neuronales Artificiales**:

<FONT COLOR="green" SIZE=3>1. Neurona artificial y Perceptrón:</FONT>  
Vimos que la unidad mínima de cómputo realiza una suma ponderada $z = \sum w_i x_i + b$ seguida de una **función de activación** $\hat{y} = f(z)$. Apilando muchas neuronas en capas obtenemos el **Perceptrón Multicapa (MLP)**.

<FONT COLOR="green" SIZE=3>2. Funciones de activación:</FONT>
<br>
Introducen la **no linealidad** que permite aprender patrones complejos. **ReLU** para capas ocultas, **Sigmoide** para clasificación binaria, **Softmax** para multiclase y **lineal** para regresión.

<FONT COLOR="green" SIZE=3>3. Backpropagation y Descenso del Gradiente:</FONT>
<br>
Es el algoritmo que permite a la red **aprender** ajustando los pesos en la dirección opuesta al gradiente de la función de pérdida: $w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$.

<FONT COLOR="green" SIZE=3>4. Keras + TensorFlow:</FONT>
<br>
El flujo `Sequential` → `compile` → `fit` → `evaluate` nos permite construir redes potentes con muy pocas líneas de código.

<FONT COLOR="red" SIZE=3>⚠️ **Limitación que abordaremos hoy:**</FONT> los MLPs **tratan cada entrada de forma independiente**. Para una imagen de 28×28 píxeles, aplanan los 784 píxeles en un vector y pierden la **estructura espacial**. Para una frase, no pueden capturar el **orden** de las palabras. Las arquitecturas que veremos hoy resuelven exactamente esto.

---

# <FONT SIZE=5 COLOR="purple"> 1. ¿Por qué necesitamos arquitecturas especializadas? </FONT>

Imagina que quieres reconocer un gato en una foto. Si usaras un MLP tradicional, tendrías que:

1. **Aplanar** la imagen (por ejemplo, $224 \times 224 \times 3 = 150{,}528$ píxeles).
2. Conectar cada píxel a cada neurona de la primera capa oculta. Con 100 neuronas, eso son **¡15 millones de parámetros solo en la primera capa!**
3. Aprender que un "gato" puede aparecer en cualquier posición de la imagen, repitiendo el aprendizaje píxel a píxel.

<br>

<center><FONT SIZE=4 COLOR="BLUE">El problema fundamental: los MLPs ignoran la estructura del dato</FONT></center>

<br>

| Tipo de dato | Estructura | ¿Qué pierde un MLP? |
|---|---|---|
| **Imagen** | Espacial (píxeles vecinos están relacionados) | La cercanía espacial; un píxel "vecino" se trata igual que uno lejano |
| **Texto / Audio** | Secuencial (el orden importa) | El orden temporal; "perro come" ≠ "come perro" |
| **Grafos** | Relacional (nodos conectados) | Las relaciones entre nodos |

<br>

<FONT COLOR="purple" SIZE=3>**La solución: sesgos inductivos (inductive biases)**</FONT>

Cada arquitectura que veremos hoy incorpora un *sesgo inductivo* específico — una **suposición sobre la estructura del dato** que reduce el número de parámetros y facilita el aprendizaje:

- **CNN:** las features útiles son **locales** y se repiten en distintas posiciones (invariancia a la traslación).
- **RNN / LSTM:** los datos llegan en **secuencia** y el estado actual depende del pasado.
- **Transformer:** todas las posiciones de la secuencia pueden **atenderse mutuamente** en paralelo.

A continuación, importamos las librerías necesarias.

In [ ]:
# Manejo de datos
import pandas as pd
import numpy  as np

# Librerías para graficar
import matplotlib.pyplot as plt
import seaborn           as sns

# Sklearn: utilidades de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import accuracy_score, classification_report, confusion_matrix

# TensorFlow + Keras: nuestro framework para Deep Learning
import tensorflow as tf
from tensorflow            import keras
from tensorflow.keras      import layers, models
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, Flatten,
                                      Conv2D, MaxPooling2D,
                                      SimpleRNN, LSTM, GRU, Embedding,
                                      BatchNormalization,
                                      MultiHeadAttention, LayerNormalization,
                                      Input, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Datasets de juguete que vienen con Keras
from tensorflow.keras.datasets import mnist, fashion_mnist, imdb

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

# <FONT SIZE=5 COLOR="purple"> 2. Redes Convolucionales (CNNs) </FONT>

## <FONT SIZE=4 COLOR="green"> 2.1 Motivación e historia </FONT>

Las **Redes Neuronales Convolucionales** (CNN, *Convolutional Neural Networks*) fueron propuestas por **Yann LeCun** a finales de los años 80. Su arquitectura clásica, **LeNet-5** (1998), ya leía dígitos manuscritos de cheques bancarios.

El verdadero despegue llegó en 2012 con **AlexNet** (Krizhevsky, Sutskever & Hinton), que pulverizó el récord en el desafío ImageNet y abrió la era moderna del *Deep Learning*.

<br>
<center><img src="https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png" alt="centered image" width="700" height="280"></center>
<center><figcaption><FONT SIZE=1 COLOR="black">Fuente: Wikimedia Commons — Arquitectura típica de una CNN</FONT></figcaption></center>
<br>

<FONT COLOR="purple" SIZE=3>**Idea clave:**</FONT> en lugar de conectar todos los píxeles con todas las neuronas, una CNN aplica **filtros (kernels) pequeños** que se **deslizan** sobre la imagen detectando patrones locales (bordes, texturas, formas).

## <FONT SIZE=4 COLOR="green"> 2.2 La operación de convolución </FONT>

Una **convolución** consiste en deslizar un pequeño filtro (matriz de pesos) sobre la imagen, multiplicar elemento a elemento y sumar:

$$ (I * K)_{i,j} = \sum_{m}\sum_{n} I_{i+m,\,j+n} \cdot K_{m,n} $$

donde $I$ es la imagen y $K$ el kernel (típicamente de $3\times 3$ o $5\times 5$).

<FONT COLOR="green" SIZE=3>**Tres propiedades clave:**</FONT>

1. **Conexiones locales:** cada neurona solo "ve" una región pequeña (el campo receptivo).
2. **Pesos compartidos:** el mismo filtro se aplica en toda la imagen → **menos parámetros**.
3. **Invariancia a la traslación:** detecta un patrón sin importar dónde aparezca.

## <FONT SIZE=4 COLOR="green"> 2.3 Pooling </FONT>

Después de la convolución solemos aplicar **pooling** (típicamente *MaxPooling*): reducir la resolución tomando el máximo (o el promedio) en ventanas pequeñas. Esto:

- **Reduce el tamaño** de los mapas de características.
- **Hace el modelo más robusto** a pequeñas traslaciones.
- **Disminuye el costo computacional**.

## <FONT SIZE=4 COLOR="green"> 2.4 Arquitectura típica de una CNN </FONT>

Una CNN clásica alterna bloques de convolución + pooling, y termina con capas densas para la clasificación:

```
INPUT → [Conv → ReLU → Pool] × N → Flatten → Dense → Softmax
```

A medida que avanzamos en la red, los filtros aprenden patrones **cada vez más abstractos**: primero bordes, luego texturas, luego partes (ojos, ruedas), finalmente objetos completos.

## <FONT SIZE=4 COLOR="green"> 2.5 Visualicemos qué hace una convolución </FONT>

Antes de entrenar una CNN, veamos visualmente el efecto de aplicar distintos filtros sobre una imagen real.

In [ ]:
# Cargamos una imagen de ejemplo del dataset MNIST
(X_mnist_train, y_mnist_train), (X_mnist_test, y_mnist_test) = mnist.load_data()

# Tomamos un dígito de ejemplo
imagen = X_mnist_train[7].astype(np.float32)
print("Forma de la imagen:", imagen.shape, " (28x28 píxeles en escala de grises)")
print("Etiqueta real:", y_mnist_train[7])

# Definimos tres filtros (kernels) clásicos de procesamiento de imágenes
kernel_bordes_v = np.array([[-1, 0, 1],
                            [-2, 0, 2],
                            [-1, 0, 1]], dtype=np.float32)   # Sobel vertical

kernel_bordes_h = np.array([[-1, -2, -1],
                            [ 0,  0,  0],
                            [ 1,  2,  1]], dtype=np.float32)  # Sobel horizontal

kernel_blur    = np.ones((3, 3), dtype=np.float32) / 9.0      # Promediado (blur)

# Función simple de convolución 2D (sin padding)
def conv2d_manual(img, kernel):
    h, w   = img.shape
    kh, kw = kernel.shape
    salida = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            salida[i, j] = (img[i:i+kh, j:j+kw] * kernel).sum()
    return salida

# Aplicamos cada filtro
img_bordes_v = conv2d_manual(imagen, kernel_bordes_v)
img_bordes_h = conv2d_manual(imagen, kernel_bordes_h)
img_blur     = conv2d_manual(imagen, kernel_blur)

# Visualizamos
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(imagen, cmap='gray');       axes[0].set_title('Original')
axes[1].imshow(img_bordes_v, cmap='gray'); axes[1].set_title('Filtro: bordes verticales')
axes[2].imshow(img_bordes_h, cmap='gray'); axes[2].set_title('Filtro: bordes horizontales')
axes[3].imshow(img_blur, cmap='gray');     axes[3].set_title('Filtro: blur (promedio)')
for ax in axes: ax.axis('off')
plt.suptitle('Efecto de distintos kernels sobre la misma imagen', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("\nLo importante: en una CNN, ¡estos kernels NO se diseñan a mano! La red los APRENDE durante el entrenamiento.")

## <FONT SIZE=4 COLOR="green"> 2.6 Ejemplo práctico: clasificación de dígitos con MNIST </FONT>

Ahora construyamos una CNN real para clasificar los dígitos manuscritos del dataset **MNIST** (10 clases: dígitos del 0 al 9).

In [ ]:
# Reescalamos los píxeles al rango [0, 1] y añadimos la dimensión del canal
X_train_cnn = X_mnist_train.astype('float32').reshape(-1, 28, 28, 1) / 255.0
X_test_cnn  = X_mnist_test.astype('float32').reshape(-1, 28, 28, 1)  / 255.0
y_train_cnn = y_mnist_train
y_test_cnn  = y_mnist_test

print("X_train shape:", X_train_cnn.shape, "  (60000 imágenes de 28x28x1)")
print("X_test  shape:", X_test_cnn.shape)
print("Clases:        ", np.unique(y_train_cnn))

# Visualicemos algunos ejemplos
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_cnn[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"Label: {y_train_cnn[i]}")
    ax.axis('off')
plt.suptitle('Ejemplos del dataset MNIST', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Definimos una CNN sencilla pero efectiva
modelo_cnn = Sequential([
    # Bloque convolucional 1
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),

    # Bloque convolucional 2
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Aplanamos y clasificamos
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')   # 10 clases (dígitos 0-9)
])

modelo_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',   # 'sparse' porque las etiquetas son enteros (no one-hot)
    metrics=['accuracy']
)

modelo_cnn.summary()

In [ ]:
# Entrenamos (pocas epochs para que sea rápido; 3-5 ya dan buenos resultados en MNIST)
historia_cnn = modelo_cnn.fit(
    X_train_cnn, y_train_cnn,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)

# Evaluamos en test
loss_cnn, acc_cnn = modelo_cnn.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"\nAccuracy en test: {acc_cnn:.4f}")

In [ ]:
# Graficamos las curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(historia_cnn.history['loss'],     label='train', linewidth=2)
axes[0].plot(historia_cnn.history['val_loss'], label='val',   linewidth=2)
axes[0].set_title('Pérdida (loss)'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(historia_cnn.history['accuracy'],     label='train', linewidth=2)
axes[1].plot(historia_cnn.history['val_accuracy'], label='val',   linewidth=2)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Veamos algunas predicciones
y_pred_proba = modelo_cnn.predict(X_test_cnn[:10], verbose=0)
y_pred       = y_pred_proba.argmax(axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test_cnn[i].reshape(28, 28), cmap='gray')
    color = 'green' if y_pred[i] == y_test_cnn[i] else 'red'
    ax.set_title(f"Real: {y_test_cnn[i]} | Pred: {y_pred[i]}", color=color)
    ax.axis('off')
plt.suptitle('Predicciones de la CNN sobre MNIST', fontweight='bold')
plt.tight_layout(); plt.show()

## <FONT SIZE=4 COLOR="blue"> 2.7 Ejercicios — CNNs </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
Explica con tus propias palabras tres ventajas de una CNN frente a un MLP cuando trabajamos con imágenes. ¿Qué quiere decir que los pesos están "compartidos"?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
Una capa `Conv2D(32, kernel_size=(3,3))` aplicada a una entrada de tamaño $28 \times 28 \times 1$:

a) ¿Cuántos parámetros entrenables tiene? (No olvides los sesgos).  
b) ¿Cuál es el tamaño de la salida (sin *padding*)?  
c) Compara ese número de parámetros con una capa densa `Dense(32)` que recibiera los $28 \times 28 = 784$ píxeles aplanados.


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
Entrena una CNN sobre el dataset **Fashion-MNIST** (10 clases de prendas de ropa). Es exactamente igual a MNIST en formato (imágenes $28 \times 28$), pero más difícil. Reporta la accuracy en test y comenta si te parece mayor o menor que en MNIST.

**Pista:** carga el dataset con `fashion_mnist.load_data()` y reutiliza la arquitectura `modelo_cnn`.

In [ ]:
# TU CÓDIGO AQUÍ:

# 1. Cargar Fashion-MNIST
(X_fm_train, y_fm_train), (X_fm_test, y_fm_test) = fashion_mnist.load_data()

class_names = ['T-shirt', 'Pantalón', 'Pullover', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

# 2. Preprocesar (reescalar y añadir canal)
X_fm_train = ...
X_fm_test  = ...

# 3. Definir una CNN similar
modelo_fm = Sequential([
    # ... completa la arquitectura ...
])

# 4. Compilar
modelo_fm.compile(...)

# 5. Entrenar 5 epochs
historia_fm = ...

# 6. Evaluar en test
loss_fm, acc_fm = ...
print(f"Accuracy Fashion-MNIST: {acc_fm:.4f}")

### <FONT SIZE=3 COLOR="blue"> Ejercicio 4 </FONT>
Investiga qué es el **padding** y el **stride** en una capa convolucional. ¿Para qué sirve `padding='same'` en Keras? Da un ejemplo numérico con una imagen de $5 \times 5$ y un kernel de $3 \times 3$.


*Escribe tu respuesta aquí:*


# <FONT SIZE=5 COLOR="purple"> 3. Redes Recurrentes (RNNs y LSTM) </FONT>

## <FONT SIZE=4 COLOR="green"> 3.1 Motivación: datos secuenciales </FONT>

Hasta ahora hemos asumido que las muestras del dataset son **independientes** entre sí. Pero en muchos problemas reales el **orden importa**:

- **Texto:** "el perro persigue al gato" ≠ "el gato persigue al perro".
- **Series de tiempo:** la temperatura de mañana depende de la de hoy y la de ayer.
- **Audio:** la onda en el instante $t$ depende de los instantes anteriores.
- **Video:** los frames están temporalmente conectados.

Las **Redes Neuronales Recurrentes** (RNN) fueron diseñadas exactamente para esto: tienen una **memoria interna** (estado oculto) que se va actualizando a medida que procesan la secuencia.

<br>
<center><img src="https://upload.wikimedia.org/wikipedia/commons/b/b5/Recurrent_neural_network_unfold.svg" alt="centered image" width="650" height="220"></center>
<center><figcaption><FONT SIZE=1 COLOR="black">Fuente: Wikimedia Commons — RNN "desenrollada" en el tiempo</FONT></figcaption></center>
<br>

## <FONT SIZE=4 COLOR="green"> 3.2 La ecuación de la RNN </FONT>

Una RNN procesa una secuencia $x_1, x_2, \dots, x_T$ paso a paso. En cada instante $t$ actualiza su **estado oculto** $h_t$:

$$ h_t = \tanh\big( W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h \big) $$

$$ \hat{y}_t = W_{hy}\, h_t + b_y $$

<FONT COLOR="green" SIZE=3>**Observación clave:**</FONT> las **mismas matrices** $W_{xh}$, $W_{hh}$, $W_{hy}$ se usan en cada paso de tiempo. La red comparte parámetros a lo largo del tiempo (igual que la CNN los comparte en el espacio).

## <FONT SIZE=4 COLOR="green"> 3.3 El problema del vanishing gradient en secuencias largas </FONT>

Las RNN simples (también llamadas *Vanilla RNN*) tienen un problema serio: cuando la secuencia es larga, el gradiente se **desvanece** (o **explota**) al propagarse hacia atrás en el tiempo. Esto hace que la red **olvide** la información de pasos lejanos.

<FONT COLOR="red" SIZE=3> **Consecuencia práctica:**</FONT> una RNN simple difícilmente conecta el principio de un texto con su final.

## <FONT SIZE=4 COLOR="green"> 3.4 LSTM: Long Short-Term Memory </FONT>

En 1997, **Hochreiter & Schmidhuber** propusieron las **LSTM**, un tipo de neurona recurrente con **compuertas** (gates) que controlan qué información:

-  **olvidar** (forget gate)
-  **memorizar** (input gate)
-  **emitir** (output gate)

<br>
<center><img src="https://upload.wikimedia.org/wikipedia/commons/3/3b/The_LSTM_cell.png" alt="centered image" width="500" height="320"></center>
<center><figcaption><FONT SIZE=1 COLOR="black">Fuente: Wikimedia Commons — Celda LSTM</FONT></figcaption></center>
<br>

Gracias a estas compuertas, las LSTM pueden **recordar información a largo plazo** y son la base de muchas aplicaciones clásicas de NLP, traducción automática (pre-Transformer) y series de tiempo.

## <FONT SIZE=4 COLOR="green"> 3.5 GRU: una alternativa más simple </FONT>

Las **GRU** (*Gated Recurrent Unit*, Cho et al. 2014) son una simplificación de las LSTM con **menos compuertas** (solo *reset* y *update*). Suelen entrenarse más rápido y rendir de forma comparable.

## <FONT SIZE=4 COLOR="green"> 3.6 Ejemplo práctico: análisis de sentimiento con IMDB </FONT>

Vamos a entrenar una LSTM para clasificar reseñas de películas del dataset **IMDB** como **positivas** o **negativas**. Es el "hello world" del procesamiento de texto.

In [ ]:
# Cargamos IMDB. Mantenemos solo las 10.000 palabras más frecuentes.
vocab_size = 10000
maxlen     = 200    # truncamos / rellenamos cada reseña a 200 palabras

(X_imdb_train, y_imdb_train), (X_imdb_test, y_imdb_test) = imdb.load_data(num_words=vocab_size)

print("Número de reseñas train:", len(X_imdb_train))
print("Número de reseñas test: ", len(X_imdb_test))
print("Primera reseña (codificada como índices de palabras):", X_imdb_train[0][:20], "...")
print("Etiqueta (0=negativa, 1=positiva):", y_imdb_train[0])

# Padding: todas las reseñas deben tener la misma longitud para entrar a la red
X_imdb_train = keras.preprocessing.sequence.pad_sequences(X_imdb_train, maxlen=maxlen)
X_imdb_test  = keras.preprocessing.sequence.pad_sequences(X_imdb_test,  maxlen=maxlen)

print("\nShape después de padding:", X_imdb_train.shape)

In [ ]:
# Decodifiquemos una reseña para verla en texto plano
word_index = imdb.get_word_index()
# Los índices 0,1,2 están reservados (padding, start, unk)
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word[0] = '<pad>'
index_to_word[1] = '<start>'
index_to_word[2] = '<unk>'

def decodifica(seq):
    return ' '.join(index_to_word.get(i, '?') for i in seq if i != 0)

print("Reseña #0 (truncada):")
print(decodifica(X_imdb_train[0])[:400], "...")
print("\nEtiqueta:", "POSITIVA " if y_imdb_train[0] == 1 else "NEGATIVA")

In [ ]:
# Construimos la red: Embedding + LSTM + Dense
modelo_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=maxlen),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')   # 1 neurona: positivo / negativo
])

modelo_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo_lstm.summary()

In [ ]:
# Entrenamos (las LSTM son más lentas que las CNN; 2-3 epochs ya rinden bien)
historia_lstm = modelo_lstm.fit(
    X_imdb_train, y_imdb_train,
    validation_split=0.2,
    epochs=3,
    batch_size=128,
    verbose=1
)

loss_lstm, acc_lstm = modelo_lstm.evaluate(X_imdb_test, y_imdb_test, verbose=0)
print(f"\nAccuracy LSTM en test IMDB: {acc_lstm:.4f}")

In [ ]:
# Probamos con dos reseñas reales del test set
for i in [0, 5]:
    proba = modelo_lstm.predict(X_imdb_test[i:i+1], verbose=0)[0, 0]
    print("-" * 80)
    print(decodifica(X_imdb_test[i])[:300], "...")
    print(f"\n>>> Etiqueta real: {'POSITIVA' if y_imdb_test[i]==1 else 'NEGATIVA'}")
    print(f">>> Predicción del modelo: {proba:.3f}  -->  {'POSITIVA' if proba>0.5 else 'NEGATIVA'}")
    print()

## <FONT SIZE=4 COLOR="blue"> 3.7 Ejercicios — RNNs y LSTM </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
Explica en una frase cada una de las tres "compuertas" de una LSTM: *forget gate*, *input gate* y *output gate*. ¿Por qué le permiten a la LSTM aprender dependencias de largo plazo mejor que una RNN simple?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
La primera capa de nuestro modelo es `Embedding(input_dim=10000, output_dim=32)`.  

a) ¿Qué hace una capa `Embedding`? ¿Por qué no podemos enchufar los índices de palabras directamente a la LSTM?  
b) ¿Cuántos parámetros tiene esta capa?  
c) ¿Qué pasaría si subiéramos `output_dim` a 128? ¿Y si lo bajáramos a 4?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
Reemplaza la capa `LSTM(64)` por una **`GRU(64)`** en el modelo de IMDB. Entrena 3 epochs y compara:
- Accuracy en test.
- Número de parámetros (`summary()`).
- Tiempo aproximado por epoch.

¿Cuál arquitectura te parece mejor para este problema?

In [ ]:
# TU CÓDIGO AQUÍ:

modelo_gru = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=maxlen),
    # ... reemplaza por GRU ...
    Dense(1, activation='sigmoid')
])

modelo_gru.compile(...)

# Imprime el resumen para ver los parámetros
modelo_gru.summary()

# Entrena 3 epochs
historia_gru = ...

# Evalúa en test
loss_gru, acc_gru = ...
print(f"\nAccuracy GRU: {acc_gru:.4f}")
print(f"Accuracy LSTM (referencia): {acc_lstm:.4f}")

### <FONT SIZE=3 COLOR="blue"> Ejercicio 4 </FONT>
**Bidireccional:** investiga qué hace la capa `Bidirectional(LSTM(...))` de Keras. ¿En qué tipo de problemas tiene sentido usarla y en cuáles **NO** (pista: piensa en tareas de predicción en tiempo real, como autocompletar o pronóstico de series de tiempo)?


*Escribe tu respuesta aquí:*


# <FONT SIZE=5 COLOR="purple"> 4. Transformers </FONT>

## <FONT SIZE=4 COLOR="green"> 4.1 La revolución de "Attention is All You Need" </FONT>

En 2017, un equipo de Google publicó el paper **"Attention is All You Need"** (Vaswani et al.) que presentaba una arquitectura nueva: el **Transformer**. La idea era romper con la dependencia secuencial de las RNN/LSTM.

<FONT COLOR="purple" SIZE=3>**Problemas que tenían las LSTM:**</FONT>

1. **No se paralelizan bien:** hay que procesar la secuencia palabra por palabra.
2. **Memoria limitada:** las dependencias muy largas siguen siendo difíciles.
3. **Lentas de entrenar** sobre datasets enormes.

<FONT COLOR="purple" SIZE=3>**La propuesta del Transformer:**</FONT>

> "¿Y si en cada paso, cada palabra **mira** directamente a todas las demás y decide cuáles son relevantes?"

Esa idea se llama **self-attention** (auto-atención).

<br>
<center><img src="https://upload.wikimedia.org/wikipedia/commons/8/8f/The-Transformer-model-architecture.png" alt="centered image" width="450" height="500"></center>
<center><figcaption><FONT SIZE=1 COLOR="black">Fuente: Wikimedia Commons — Arquitectura original del Transformer</FONT></figcaption></center>
<br>

## <FONT SIZE=4 COLOR="green"> 4.2 Self-Attention: el corazón del Transformer </FONT>

La idea central: cada palabra del input se transforma en **tres vectores**:

- **Query (Q):** "¿qué estoy buscando?"
- **Key (K):** "¿qué información ofrezco?"
- **Value (V):** "este es mi contenido"

La atención se calcula así:

$$ \text{Attention}(Q, K, V) = \text{softmax}\!\left(\dfrac{QK^T}{\sqrt{d_k}}\right) V $$

<FONT COLOR="green" SIZE=3>**Interpretación intuitiva:**</FONT>
1. Cada palabra (con su $Q$) hace producto punto con las $K$ de todas las demás → da un **score** de relevancia.
2. Aplicamos softmax → obtenemos **pesos de atención** que suman 1.
3. Tomamos un promedio ponderado de los $V$ → cada palabra recibe un nuevo vector que mezcla información de las demás según la relevancia.

## <FONT SIZE=4 COLOR="green"> 4.3 Multi-Head Attention </FONT>

En lugar de calcular **una sola atención**, el Transformer calcula **varias en paralelo** (las "cabezas") y las concatena. Cada cabeza puede aprender a fijarse en un aspecto distinto de la relación entre palabras (sintaxis, semántica, correferencias, etc.).

## <FONT SIZE=4 COLOR="green"> 4.4 Modelos famosos basados en Transformers </FONT>

| Modelo | Año | Características |
|---|---|---|
| **BERT** | 2018 (Google) | Solo *encoder*. Comprende contexto bidireccional. Ideal para clasificación, NER, QA. |
| **GPT-2/3/4** | 2019+ (OpenAI) | Solo *decoder*. Generativo, predice la siguiente palabra. Base de ChatGPT. |
| **T5** | 2019 (Google) | Encoder-decoder. Todo se formula como "text-to-text". |
| **Vision Transformer (ViT)** | 2020 | Aplica el Transformer a imágenes (parches en vez de palabras). |
| **Claude / Llama / Gemini** | 2023+ | LLMs modernos de gran escala, descendientes de GPT/T5. |

<FONT COLOR="red" SIZE=3>**Importante:**</FONT> entrenar un Transformer grande desde cero requiere **toneladas de datos y GPUs**. En la práctica casi siempre se usa **transfer learning** con un modelo pre-entrenado (BERT, GPT-2, etc.) que después se ajusta a la tarea específica.

## <FONT SIZE=4 COLOR="green"> 4.5 Implementemos un mini-Transformer </FONT>

Para entender la idea de forma palpable, construyamos un **bloque Transformer sencillo** y úsalo para clasificar las mismas reseñas de IMDB. No buscamos batir al estado del arte; buscamos **comprender la arquitectura**.

In [ ]:
# Un bloque Transformer sencillo: Multi-Head Attention + Feed Forward, con residual + LayerNorm
class BloqueTransformer(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att     = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn     = Sequential([Dense(ff_dim, activation='relu'),
                                   Dense(embed_dim)])
        self.norm1   = LayerNormalization(epsilon=1e-6)
        self.norm2   = LayerNormalization(epsilon=1e-6)
        self.drop1   = Dropout(rate)
        self.drop2   = Dropout(rate)

    def call(self, inputs, training=False):
        # 1. Self-Attention con conexión residual + normalización
        attn_output = self.att(inputs, inputs)              # Q = K = V = inputs
        attn_output = self.drop1(attn_output, training=training)
        out1        = self.norm1(inputs + attn_output)
        # 2. Feed-Forward con conexión residual + normalización
        ffn_output  = self.ffn(out1)
        ffn_output  = self.drop2(ffn_output, training=training)
        return self.norm2(out1 + ffn_output)


# Embedding que combina token + posición
# (los Transformers no tienen recurrencia, así que necesitan que les digamos "dónde" está cada palabra)
class TokenYPosicionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb   = Embedding(input_dim=maxlen,     output_dim=embed_dim)

    def call(self, x):
        positions  = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

In [ ]:
# Construimos el modelo completo: Embedding -> Transformer -> Pooling -> Dense
embed_dim = 32       # dimensión del embedding
num_heads = 2        # número de cabezas de atención
ff_dim    = 32       # tamaño de la capa feed-forward interna

inputs   = Input(shape=(maxlen,))
x        = TokenYPosicionEmbedding(maxlen, vocab_size, embed_dim)(inputs)
x        = BloqueTransformer(embed_dim, num_heads, ff_dim)(x)
x        = GlobalAveragePooling1D()(x)            # promedio sobre la dimensión temporal
x        = Dropout(0.1)(x)
x        = Dense(20, activation='relu')(x)
x        = Dropout(0.1)(x)
outputs  = Dense(1, activation='sigmoid')(x)

modelo_tfm = Model(inputs=inputs, outputs=outputs)

modelo_tfm.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

modelo_tfm.summary()

In [ ]:
# Entrenamos
historia_tfm = modelo_tfm.fit(
    X_imdb_train, y_imdb_train,
    validation_split=0.2,
    epochs=3,
    batch_size=128,
    verbose=1
)

loss_tfm, acc_tfm = modelo_tfm.evaluate(X_imdb_test, y_imdb_test, verbose=0)
print(f"\nAccuracy Mini-Transformer en IMDB: {acc_tfm:.4f}")
print(f"Accuracy LSTM (referencia):         {acc_lstm:.4f}")

## <FONT SIZE=4 COLOR="blue"> 4.6 Ejercicios — Transformers </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
Explica con tus propias palabras qué hace el mecanismo de **self-attention**. ¿Por qué se dice que "cada palabra mira a todas las demás"? Da un ejemplo concreto con una frase corta como *"el banco al lado del río"*: ¿cómo crees que la atención podría ayudar a desambiguar la palabra "banco"?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
Compara LSTM y Transformer en estos aspectos:

| Aspecto | LSTM | Transformer |
|---|---|---|
| ¿Procesa la secuencia paso a paso o en paralelo? | | |
| ¿Cómo maneja dependencias a larga distancia? | | |
| ¿Necesita información explícita de la posición? | | |
| ¿Cuánta memoria/GPU requiere para secuencias largas? | | |


*Completa la tabla anterior aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
En el modelo construido arriba, prueba aumentar `num_heads` de 2 a **4** y `ff_dim` de 32 a **64**. Entrena 3 epochs y reporta:
- Accuracy en test.
- Número total de parámetros.

¿Mejoró la accuracy? ¿Cuál es el costo?

In [ ]:
# TU CÓDIGO AQUÍ:

embed_dim_2 = 32
num_heads_2 = 4      # <-- modifica
ff_dim_2    = 64     # <-- modifica

inputs2  = Input(shape=(maxlen,))
x2       = TokenYPosicionEmbedding(maxlen, vocab_size, embed_dim_2)(inputs2)
x2       = BloqueTransformer(embed_dim_2, num_heads_2, ff_dim_2)(x2)
x2       = GlobalAveragePooling1D()(x2)
x2       = Dropout(0.1)(x2)
x2       = Dense(20, activation='relu')(x2)
outputs2 = Dense(1, activation='sigmoid')(x2)

modelo_tfm2 = Model(inputs=inputs2, outputs=outputs2)
modelo_tfm2.compile(...)
modelo_tfm2.summary()

historia_tfm2 = ...
loss_tfm2, acc_tfm2 = ...
print(f"\nAccuracy con 4 heads / ff_dim=64: {acc_tfm2:.4f}")

### <FONT SIZE=3 COLOR="blue"> Ejercicio 4 — Conceptual </FONT>
Los Transformers modernos (BERT, GPT) tienen **cientos de millones a billones de parámetros**. Investiga:

a) ¿Aproximadamente cuántos parámetros tiene GPT-3? ¿Y GPT-4 (estimación pública)?  
b) ¿Qué es el **fine-tuning** vs el **pre-training**? ¿Por qué casi nadie entrena un Transformer grande desde cero?  
c) ¿Qué es un **prompt** y cómo se relaciona con todo lo que acabas de aprender?


*Escribe tu respuesta aquí:*


# <FONT SIZE=5 COLOR="purple"> 5. Regularización Avanzada </FONT>

## <FONT SIZE=4 COLOR="green"> 5.1 Repaso: el problema del sobreajuste </FONT>

Las redes profundas tienen **millones de parámetros**. Esto las hace muy expresivas, pero también muy propensas al **sobreajuste (overfitting)**: aprenden de memoria el training set y generalizan mal a datos nuevos.

<FONT COLOR="green" SIZE=3>**Diagnóstico visual:**</FONT>

- Si `train_loss` baja pero `val_loss` sube → **overfitting**.
- Si ambas se mantienen altas → **underfitting**.

Ya conocimos una técnica: **Dropout**. Pero hay más, y muy importantes en redes modernas:

| Técnica | ¿Qué hace? | ¿Cuándo usarla? |
|---|---|---|
| **Batch Normalization** | Normaliza las activaciones intermedias | Casi siempre, en redes profundas |
| **Early Stopping** | Detiene el entrenamiento cuando la val_loss deja de bajar | Siempre que entrenes muchas epochs |
| **Data Augmentation** | Genera variaciones artificiales de los datos | Cuando tienes pocos datos (especialmente imágenes) |
| **L1 / L2 (weight decay)** | Penaliza pesos grandes | Modelos pequeños o con muchos parámetros |
| **Dropout** | Apaga neuronas aleatoriamente | Capas densas; ya lo vimos en la sesión 6 |

## <FONT SIZE=4 COLOR="green"> 5.2 Batch Normalization </FONT>

Propuesta por **Ioffe & Szegedy (2015)**. La idea es **normalizar las activaciones** de cada capa para que tengan media 0 y desviación estándar 1 **dentro de cada mini-batch**:

$$ \hat{x}_i = \dfrac{x_i - \mu_B}{\sqrt{\sigma^2_B + \epsilon}}, \quad\quad y_i = \gamma \hat{x}_i + \beta $$

donde $\gamma$ y $\beta$ son parámetros **aprendibles** que permiten que la red recupere la escala original si lo necesita.

<FONT COLOR="green" SIZE=3>**Beneficios:**</FONT>

1. **Permite usar learning rates más altos** → entrenamiento más rápido.
2. **Estabiliza el entrenamiento** (reduce el *internal covariate shift*).
3. **Actúa como regularizador** (efecto similar a Dropout en menor escala).
4. **Hace al modelo menos sensible** a la inicialización de pesos.

En Keras es tan simple como añadir `BatchNormalization()` entre capas:

```python
Dense(64), BatchNormalization(), Activation('relu')
```

## <FONT SIZE=4 COLOR="green"> 5.3 Early Stopping </FONT>

Idea muy sencilla pero efectiva: **detén el entrenamiento cuando la pérdida de validación deje de mejorar**. Así evitas entrenar de más y caer en overfitting.

En Keras usamos el *callback* `EarlyStopping`:

```python
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
modelo.fit(..., callbacks=[es])
```

- `patience=5`: tolera 5 epochs sin mejora antes de detenerse.
- `restore_best_weights=True`: al detenerse, **vuelve** a los mejores pesos vistos.

## <FONT SIZE=4 COLOR="green"> 5.4 Data Augmentation </FONT>

Cuando tienes pocos datos (típicamente con imágenes), puedes **generar variaciones artificiales** de las muestras existentes: rotaciones, zoom, espejados, cambios de brillo, etc. Esto:

- **Multiplica el tamaño efectivo** del dataset.
- **Enseña a la red invariancias** útiles (un gato sigue siendo un gato si lo rotas 15°).
- **Reduce drásticamente el overfitting**.

Para imágenes, Keras tiene `ImageDataGenerator` y la versión moderna `tf.keras.layers.RandomFlip`, `RandomRotation`, `RandomZoom`, etc.

## <FONT SIZE=4 COLOR="green"> 5.5 Demo: comparemos un modelo SIN y CON regularización avanzada </FONT>

Usaremos un subconjunto **pequeño** del dataset MNIST (solo 2.000 imágenes para entrenar) para que el sobreajuste se note más, y compararemos dos modelos:
- **Modelo A:** sin regularización avanzada.
- **Modelo B:** con BatchNorm + Dropout + EarlyStopping + Data Augmentation.

In [ ]:
# Tomamos un subconjunto pequeño para forzar el overfitting
n_train = 2000
X_small = X_train_cnn[:n_train]
y_small = y_train_cnn[:n_train]
print("Tamaño del training set reducido:", X_small.shape)

# === MODELO A: sin regularización avanzada ===
modelo_A = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])
modelo_A.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

hist_A = modelo_A.fit(X_small, y_small,
                     validation_split=0.2,
                     epochs=20, batch_size=64, verbose=0)
loss_A, acc_A = modelo_A.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"\nModelo A (sin regularización avanzada) — Accuracy test: {acc_A:.4f}")

In [ ]:
# === MODELO B: con BatchNorm + Dropout + EarlyStopping + Data Augmentation ===

# 1. Capas de data augmentation aplicadas en cada batch
data_augmentation = Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    layers.RandomTranslation(0.05, 0.05),
])

modelo_B = Sequential([
    data_augmentation,                                            # Data Augmentation
    Conv2D(32, (3, 3), input_shape=(28, 28, 1)),
    BatchNormalization(),                                         # BatchNorm
    layers.Activation('relu'),
    MaxPooling2D(),
    Conv2D(64, (3, 3)),
    BatchNormalization(),                                         # BatchNorm
    layers.Activation('relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(128),
    BatchNormalization(),
    layers.Activation('relu'),
    Dropout(0.4),                                                 # Dropout
    Dense(10, activation='softmax')
])
modelo_B.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 2. Early Stopping
es = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

hist_B = modelo_B.fit(X_small, y_small,
                     validation_split=0.2,
                     epochs=20, batch_size=64,
                     callbacks=[es],
                     verbose=0)
loss_B, acc_B = modelo_B.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"Modelo B (con regularización avanzada) — Accuracy test: {acc_B:.4f}")
print(f"\nÉpocas entrenadas: A={len(hist_A.history['loss'])} | B={len(hist_B.history['loss'])} (B paró antes por EarlyStopping)")

In [ ]:
# Comparemos visualmente las curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist_A.history['loss'],     label='A train', linewidth=2, linestyle='--')
axes[0].plot(hist_A.history['val_loss'], label='A val',   linewidth=2, linestyle='--')
axes[0].plot(hist_B.history['loss'],     label='B train', linewidth=2)
axes[0].plot(hist_B.history['val_loss'], label='B val',   linewidth=2)
axes[0].set_title('Pérdida — A (sin reg) vs B (con reg avanzada)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_A.history['accuracy'],     label='A train', linewidth=2, linestyle='--')
axes[1].plot(hist_A.history['val_accuracy'], label='A val',   linewidth=2, linestyle='--')
axes[1].plot(hist_B.history['accuracy'],     label='B train', linewidth=2)
axes[1].plot(hist_B.history['val_accuracy'], label='B val',   linewidth=2)
axes[1].set_title('Accuracy — A vs B')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\nResumen final:")
print(f"  Modelo A (sin regularización avanzada): {acc_A:.4f}")
print(f"  Modelo B (con regularización avanzada): {acc_B:.4f}")
print(f"  Mejora absoluta: {(acc_B - acc_A)*100:.2f} puntos porcentuales")

<FONT COLOR="purple" SIZE=3>**Observación esperada:**</FONT> el Modelo A muestra una **brecha enorme** entre train y val (sobreajuste), mientras que el Modelo B mantiene las dos curvas mucho más cerca y, además, tiende a generalizar mejor en test.

## <FONT SIZE=4 COLOR="green"> 5.6 Veamos las imágenes aumentadas </FONT>

In [ ]:
# Visualicemos qué hace la data augmentation
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
ejemplo = X_small[0:1]
for i, ax in enumerate(axes.flat):
    img_aug = data_augmentation(ejemplo, training=True)
    ax.imshow(img_aug[0].numpy().reshape(28, 28), cmap='gray')
    ax.set_title(f'Aumentada {i+1}')
    ax.axis('off')
plt.suptitle('La MISMA imagen vista 10 veces tras Data Augmentation\n(rotación, zoom, traslación)', fontweight='bold')
plt.tight_layout(); plt.show()

## <FONT SIZE=4 COLOR="blue"> 5.7 Ejercicios — Regularización Avanzada </FONT>

### <FONT SIZE=3 COLOR="blue"> Ejercicio 1 </FONT>
Para cada una de las siguientes situaciones, indica **qué técnica de regularización** usarías y por qué:

a) Tienes 200 imágenes etiquetadas para clasificar plantas y notas que la val_accuracy se queda en 60%.  
b) Tu red de 10 capas entrena lentísimo y la loss oscila mucho.  
c) Entrenas durante 100 epochs y al final notas que el mejor modelo era el de la epoch 30.  
d) Tu MLP con 1 millón de parámetros tiene `train_acc=0.99` y `val_acc=0.72`.


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 2 </FONT>
Modifica el **Modelo A** para que incorpore **solo Early Stopping** (sin BatchNorm ni Data Augmentation). Entrena hasta 30 epochs con `patience=5`. ¿Logras superar la accuracy original del modelo A?

In [ ]:
# TU CÓDIGO AQUÍ:

# 1. Define el modelo (idéntico al A original)
modelo_A_es = Sequential([
    # ... copia la arquitectura del modelo A ...
])

modelo_A_es.compile(...)

# 2. Crea el callback de Early Stopping
es_callback = EarlyStopping(...)

# 3. Entrena con ese callback (hasta 30 epochs)
hist_A_es = ...

# 4. Evalúa
loss_A_es, acc_A_es = ...
print(f"Modelo A original:        {acc_A:.4f}")
print(f"Modelo A + EarlyStopping: {acc_A_es:.4f}")
print(f"Épocas usadas: {len(hist_A_es.history['loss'])}")

### <FONT SIZE=3 COLOR="blue"> Ejercicio 3 </FONT>
Investiga la diferencia entre **Batch Normalization** y **Layer Normalization**. 

a) ¿Sobre qué dimensión normaliza cada una?  
b) ¿Por qué los Transformers usan **LayerNorm** en lugar de **BatchNorm**?


*Escribe tu respuesta aquí:*


### <FONT SIZE=3 COLOR="blue"> Ejercicio 4 — Reto </FONT>
Diseña y entrena tu **propia CNN** para Fashion-MNIST que combine **todas** las técnicas de esta sesión:
1. Al menos 2 bloques Conv2D + BatchNorm + ReLU + MaxPooling.
2. Data Augmentation al inicio.
3. Dropout en las capas densas.
4. Early Stopping durante el entrenamiento.

**Meta:** superar el **88% de accuracy en test** entrenando un máximo de **25 epochs**.

In [ ]:
# TU CÓDIGO AQUÍ:

# 1. Preparamos Fashion-MNIST (si no lo hiciste antes)
(X_fm_train, y_fm_train), (X_fm_test, y_fm_test) = fashion_mnist.load_data()
X_fm_train = X_fm_train.astype('float32').reshape(-1, 28, 28, 1) / 255.0
X_fm_test  = X_fm_test.astype('float32').reshape(-1, 28, 28, 1)  / 255.0

# 2. Define la arquitectura combinando TODAS las técnicas
data_aug = Sequential([
    # ... rotación, zoom, flip horizontal, etc. ...
])

modelo_reto = Sequential([
    data_aug,
    # ... Conv2D + BatchNorm + ReLU + MaxPool ...
    # ... Otro bloque convolucional ...
    Flatten(),
    Dense(...),
    BatchNormalization(),
    # ... Activación, Dropout ...
    Dense(10, activation='softmax')
])

modelo_reto.compile(...)

# 3. Early stopping
es_reto = EarlyStopping(...)

# 4. Entrena
hist_reto = ...

# 5. Evalúa
loss_reto, acc_reto = ...
print(f"\nAccuracy en Fashion-MNIST test: {acc_reto:.4f}")
print("¿Superaste 0.88?" , "🎉 Sí" if acc_reto > 0.88 else "🔁 Aún no, sigue ajustando")

---
# <FONT SIZE=5 COLOR="purple"> 6. Conclusiones de la sesión </FONT>

En esta sesión cruzamos la línea del *Deep Learning* moderno:

<FONT COLOR="green" SIZE=3>1. CNNs:</FONT> aprendimos que la **convolución** y el **pooling** explotan la estructura espacial de las imágenes, comparten pesos y son la base de la visión por computador moderna.

<FONT COLOR="green" SIZE=3>2. RNNs y LSTM:</FONT> entendimos cómo una red con **memoria** procesa secuencias paso a paso, por qué las RNN simples sufren *vanishing gradient*, y cómo las **LSTM** lo resuelven con sus compuertas.

<FONT COLOR="green" SIZE=3>3. Transformers:</FONT> exploramos el mecanismo de **self-attention**, vimos cómo permite procesar secuencias **en paralelo** y por qué es la arquitectura detrás de **BERT, GPT, ChatGPT** y prácticamente toda la IA generativa actual.

<FONT COLOR="green" SIZE=3>4. Regularización avanzada:</FONT> incorporamos **Batch Normalization** para estabilizar el entrenamiento, **Early Stopping** para evitar entrenar de más, y **Data Augmentation** para multiplicar artificialmente nuestros datos.

---

<FONT SIZE=3 COLOR="purple">**¿Qué viene después?**</FONT>

- **Transfer Learning** y modelos pre-entrenados (ResNet, EfficientNet, BERT, GPT) — aprovechar lo aprendido por otros.
- **Modelos generativos:** GANs, VAEs y modelos de difusión (Stable Diffusion, DALL·E).
- **Reinforcement Learning:** agentes que aprenden de la interacción (AlphaGo, robótica, juegos).
- **Modelos multimodales:** texto + imagen + audio (GPT-4V, CLIP, Whisper).
- **MLOps:** llevar estos modelos a producción de forma robusta y escalable.

¡Hemos recorrido un camino enorme! De una sola neurona a la arquitectura detrás de ChatGPT.